# Calcination imputation sensitivity

Run steps 01–03 and 05 first. Open this notebook from the workflow folder and run all cells. Table S1 compares training-set mean imputation with complete-case deletion on the same held-out records.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

In [ ]:



POINT_ID = "Data point number"  # Unique record identifier column.
CALCINATION = ["Calc T", "Calc D"]  # Calcination temperature and duration columns.
FEATURES = [  # Predictor columns for the STY comparison.
    "wt% Cu", "wt% Support1", "χ Support1", "wt% Support2", "χ Support2",
    "wt% Support3", "χ Support3", "wt% Promoter", "χ Promoter",
    "Synthesis Method", "Calc T", "Calc D", "H2/CO2", "GHSV",
    "P", "T", "SBET", "SACu", "dCu",
]

In [ ]:
def indexed_records(frame, source):  # Validate point identifiers and use them as the row index.
    if frame[POINT_ID].isna().any() or not frame[POINT_ID].is_unique:
        raise ValueError(f"{source} must have present, unique data point numbers.")
    return frame.set_index(POINT_ID)


def read_inputs(data_dir):  # Load the modeling inputs and recover the exact STY split and fitted settings.
    dataset_path = data_dir / "derived/intermediate" / "dataset.xlsx"  # Path to the required dataset workbook.
    results_path = data_dir / "STY_results" / "STY_results.xlsx"  
    raw = indexed_records(pd.read_excel(dataset_path, sheet_name="data"), dataset_path.name)  # Source records before modeling preparation.

    with pd.ExcelFile(results_path) as results:  
        saved = indexed_records(results.parse("predictions"), results_path.name)  
        params = results.parse("best_params", index_col=0)["Value"].to_dict()  # Estimator hyperparameters.

    missing_ids = saved.index.difference(raw.index)  
    if len(missing_ids):
        raise ValueError(f"Data points in the saved STY results are missing from dataset.xlsx: {missing_ids.tolist()}")
    retained = raw.loc[saved.index, FEATURES + ["STY"]].copy()  
    methods = pd.read_excel(data_dir / "derived" / "dataset.xlsx", sheet_name="encoding")  # Synthesis method categories or their codes.
    method_codes = methods.set_index("Synthesis Method")["Code"]  
    retained["Synthesis Method"] = retained["Synthesis Method"].astype("string").str.strip().map(method_codes)
    if retained["Synthesis Method"].isna().any():
        raise ValueError("Some retained synthesis methods have no code in the encoding sheet.")

    retained = retained.apply(pd.to_numeric).astype(float)
    required = retained.columns.difference(CALCINATION)  
    if not np.isfinite(retained[required].to_numpy()).all():
        raise ValueError("Non-calcination features and STY must contain finite values.")
    if not np.isclose(retained["STY"], saved["STY"], rtol=0, atol=1e-8).all():
        raise ValueError("STY values differ between dataset.xlsx and the saved STY results.")

    train, held_out = train_test_split(retained.index, test_size=0.20, random_state=0)  # Development records or their indices; Indices of the held-out test records.
    split = pd.Series("Test", index=retained.index)  # Training and held-out partition assignments.
    split.loc[train] = "Train"
    if not split.eq(saved["Split"]).all():
        raise ValueError("The split does not match the saved STY results.")

    for name in ("max_depth", "min_child_weight", "n_estimators"):  
        params[name] = int(params[name])
    return retained[FEATURES], retained["STY"], train, held_out, params

In [ ]:
def compare_imputation(X, y, train, held_out, params):  # Compare two calcination treatments on the same complete held-out rows.
    complete = X[CALCINATION].notna().all(axis=1)  # Rows with both calcination fields observed.
    complete_train = train[complete.loc[train].to_numpy()]  # Complete train.
    complete_test = held_out[complete.loc[held_out].to_numpy()]  # Complete test.
    means = np.floor(X.loc[train, CALCINATION].mean() + 0.5)  # Calculated means for imputation or plotting.
    if means.isna().any():
        raise ValueError("Each calcination feature needs observed values in the training set.")
    if len(complete_train) == 0 or len(complete_test) < 2:
        raise ValueError("The complete-case comparison needs training records and at least two test records.")

    test_values = X.loc[complete_test]  
    test_targets = y.loc[complete_test]  
    rows = [] 
    for method, train_values in ( 
        ("Training-set mean imputation", X.loc[train].fillna(means)),
        ("Complete-case deletion", X.loc[complete_train]),
    ):
        model = XGBRegressor(objective="reg:squarederror", tree_method="hist", random_state=0, n_jobs=1, **params)  # Estimator for the current task.
        model.fit(train_values, y.loc[train_values.index])
        predicted = np.clip(model.predict(test_values), 0, None)  # Predicted response values.
        rows.append({
            "Method": method,
            "Training records": len(train_values),
            "Held-out records": len(test_values),
            "R2": r2_score(test_targets, predicted),
            "RMSE": np.sqrt(mean_squared_error(test_targets, predicted)),
        })

    return pd.DataFrame(rows), means, int((~complete).sum())

In [ ]:
data_dir = Path.cwd()  # Workflow folder containing the generated input workbooks and STY results.
X, y, train, held_out, params = read_inputs(data_dir)  # Predictor matrix; Observed response; Development records or their indices; Indices of the held-out test records; Estimator hyperparameters.
metrics, means, incomplete = compare_imputation(X, y, train, held_out, params)  # Performance metrics; Calculated means for imputation or plotting; Incomplete.

print(f"Records missing calcination temperature or duration: {incomplete} of {len(X)}")
print(f"Rounded training means: Calc T = {means['Calc T']:.0f}, Calc D = {means['Calc D']:.0f}")
print("RMSE units: mg MeOH h^-1 g_cat^-1")
print(metrics.to_string(index=False, formatters={"R2": "{:.3f}".format, "RMSE": "{:.2f}".format}))  # Print Table S1, the comparison of calcination imputation and complete-case deletion.